In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
aircraft_schema = StructType([
    StructField("hex", StringType(), True),
    StructField("flight",StringType(),True),
    StructField("t",StringType(),True),
    StructField("alt_baro",IntegerType(),True),
    StructField("gs",DoubleType(),True),
    StructField("lat",DoubleType(),True),
    StructField("lon",DoubleType(),True),
    StructField("squawk",StringType(),True),
    StructField("track",    DoubleType(),  True),   # heading 0-360°  ← turn_rate needs this
    StructField("baro_rate",IntegerType(), True),   # ft/min from ADS-B ← replaces your computed delta
    StructField("category", StringType(),  True),   # A1/A3/A5 aircraft class
    StructField("nic",      IntegerType(), True),   # nav integrity — filter spoofed pings
    StructField("seen",     DoubleType(),  True),   # secs since last update — filter stale
])
    
my_schema = StructType([
    StructField("ac", ArrayType(aircraft_schema), True),
     StructField("now", DoubleType(), True)
])


In [0]:
# raw_stream_df = spark.readStream.format('json') \
#     .option("multiLine", True) \
#     .schema(my_schema) \
#     .option("cleanSource", "archive") \
#     .option("sourceArchiveDir", archive_path) \
#     .load(source_path)

# for now remove the Archive
source_path = "dbfs:/Volumes/pyspark/stream/sparkstreaming/sparksource"
raw_stream_df = spark.readStream.format('json') \
    .option("multiLine", True) \
    .schema(my_schema) \
    .load(source_path)

In [0]:
flattened_df = raw_stream_df.select(
    explode(col("ac")).alias("aircraft"), 
    col("now").alias("api_timestamp")
).select(
    "aircraft.*", 
    "api_timestamp"
).withColumn("processing_time", current_timestamp())

In [0]:
from pyspark.sql.functions import col

extracted_df = flattened_df.filter(
    (col("hex").isNotNull()) &
    (col("lat").isNotNull()) &
    (col("lon").isNotNull()) &
    (col("alt_baro").isNotNull()) &
    (col("nic") >= 6) &            # NIC < 6 = position uncertainty > 0.6 nmi
    (col("seen") <= 15)            # ignore pings older than 15 seconds
)

In [0]:
from pyspark.sql.functions import expr

clean_df = extracted_df.withColumn(
    "event_time",
    expr("current_timestamp() + (CAST(rand()*50 AS INT)) * INTERVAL 1 SECOND")
)

In [0]:

import time
unique_id = int(time.time())
checkpoint_path = f"dbfs:/Volumes/pyspark/stream/sparkstreaming/checkpoints/congestion_{unique_id}"

query = clean_df.writeStream \
    .format("memory") \
    .queryName("debug_view") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .outputMode("append") \
    .start()

# 2. Wait for the stream to finish processing
query.awaitTermination()
display(spark.sql("SELECT * FROM debug_view"))

In [0]:
state_df = clean_df.withColumn(
    "state",
    when(col("track").isNull(), "UNKNOWN")
    .when(col("baro_rate") > 500, "CLIMB")
    .when((col("baro_rate") < -300) & (col("alt_baro") < 10000), "APPROACH")
    .when(col("baro_rate") < -300, "DESCENT")
    .when(col("alt_baro") < 1000, "TAXI_OR_TAKEOFF")
    .otherwise("CRUISE")
)

In [0]:
# delay_df = state_df.withColumn(
#     "is_delayed",
#     when(col("gs") < 50, 1).otherwise(0)
# )

agg_df = state_df.groupBy(
    window(col("event_time"), "2 minutes", "30 seconds"),  # sliding window
    col("hex"),       # use hex (ICAO24) — flight callsign can be null
    col("state")
).agg(
    avg("gs").alias("avg_speed"),
    avg("alt_baro").alias("avg_altitude"),
    avg("baro_rate").alias("avg_vert_rate"),
    count("*").alias("ping_count"),
    first("lat").alias("seg_start_lat"),   # for geo metadata
    last("lat").alias("seg_end_lat"),
    first("lon").alias("seg_start_lon"),
    last("lon").alias("seg_end_lon"),
    first("category").alias("aircraft_category")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def process_batch(batch_df, batch_id):

    # ── Turn rate via lag (only works on static DataFrame inside foreachBatch) ──
    w_spec = Window.partitionBy("hex").orderBy("event_time")

    batch_df = batch_df \
        .withColumn("prev_track",      F.lag("track").over(w_spec)) \
        .withColumn("dt_sec",
            F.unix_timestamp("event_time") - F.lag(F.unix_timestamp("event_time")).over(w_spec)
        ) \
        .withColumn("heading_delta",
            F.when(
                F.col("prev_track").isNotNull() & (F.col("dt_sec") > 0),
                F.abs(F.col("track") - F.col("prev_track"))
            ).otherwise(F.lit(None))
        ) \
        .withColumn("heading_delta",                          # shortest arc fix
            F.when(F.col("heading_delta") > 180, 360.0 - F.col("heading_delta"))
             .otherwise(F.col("heading_delta"))
        ) \
        .withColumn("turn_rate",
            F.when(F.col("heading_delta").isNotNull(),
                   F.col("heading_delta") / F.col("dt_sec"))
             .otherwise(F.lit(0.0))
        )

    # ── Upgrade state to include TURN (needs turn_rate from above) ─────────────
    batch_df = batch_df.withColumn(
        "state",
        F.when(F.col("track").isNull(),                                    "UNKNOWN")
         .when(F.col("turn_rate") > 3.0,                                   "TURN")
         .when(F.col("baro_rate") > 500,                                   "CLIMB")
         .when((F.col("baro_rate") < -300) & (F.col("alt_baro") < 10000), "APPROACH")
         .when(F.col("baro_rate") < -300,                                  "DESCENT")
         .when(F.col("alt_baro") < 1000,                                   "TAXI_OR_TAKEOFF")
         .otherwise("CRUISE")
    )

    # ── Your existing agg_df logic — extended with turn_rate + trajectory ──────
    agg_df = batch_df.groupBy(
        F.window(F.col("event_time"), "2 minutes", "30 seconds"),
        F.col("hex"),
        F.col("state")
    ).agg(
        F.avg("gs").alias("avg_speed"),
        F.avg("alt_baro").alias("avg_altitude"),
        F.avg("baro_rate").alias("avg_vert_rate"),
        F.avg("turn_rate").alias("avg_turn_rate"),
        F.max("turn_rate").alias("max_turn_rate"),
        F.count("*").alias("ping_count"),
        F.first("lat").alias("seg_start_lat"),
        F.last("lat").alias("seg_end_lat"),
        F.first("lon").alias("seg_start_lon"),
        F.last("lon").alias("seg_end_lon"),
        F.first("category").alias("aircraft_category"),
        F.collect_list(
            F.struct("lat", "lon", "alt_baro", "gs", "track", "turn_rate")
        ).alias("trajectory")
    )
    agg_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .save("dbfs:/Volumes/pyspark/stream/sparkstreaming/segments")

In [0]:
import time
unique_id = int(time.time())
checkpoint_path = f"dbfs:/Volumes/pyspark/stream/sparkstreaming/checkpoints/congestion_{unique_id}"

query = state_df \
    .writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .start()

query.awaitTermination()

In [0]:
df = spark.read.format("delta").load("dbfs:/Volumes/pyspark/stream/sparkstreaming/segments")

# display(df)

In [0]:
df = df.select("hex", "seg_start_lat", "seg_start_lon", 
               "seg_end_lat", "seg_end_lon", "window",
               "aircraft_category", "state", "trajectory")

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, FloatType

SEQ_LEN = 30
FEATURES = ["lat", "lon", "alt_baro", "gs", "track", "turn_rate"]

@pandas_udf(ArrayType(FloatType()))
def trajectory_to_flat(traj_series: pd.Series) -> pd.Series:
    results = []
    for traj in traj_series:
        matrix = np.zeros((SEQ_LEN, 6), dtype=np.float32)
        for i, ping in enumerate(traj[:SEQ_LEN]):
            matrix[i] = [ping["lat"], ping["lon"], ping["alt_baro"], 
             ping["gs"], ping["track"], ping["turn_rate"]]
        # returns flattened (30*6 = 180,) — reshape back later
        results.append(matrix.flatten().tolist())
    return pd.Series(results)

df = df.withColumn("traj_flat", trajectory_to_flat("trajectory"))

In [0]:
df.write.mode("overwrite").parquet("dbfs:/Volumes/pyspark/stream/sparkstreaming/donwloads/segments")

In [0]:
import os
os.environ["HF_HOME"] = "/local_disk0/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/local_disk0/hf_cache"

In [0]:
%pip install uni2ts einops

In [0]:
import os
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/tmp/hf_cache"
os.makedirs("/tmp/hf_cache", exist_ok=True)

from uni2ts.model.moirai import MoiraiModule

model = MoiraiModule.from_pretrained("Salesforce/moirai-1.0-R-small")
model.save_pretrained("/dbfs/tmp/moirai-small")

In [0]:
import torch
import numpy as np
import pandas as pd
from uni2ts.model.moirai import MoiraiModule
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, FloatType

SEQ_LEN = 30
D_OUT = 256

# Cache so model loads once per executor, not per row
_model_cache = {}

def get_model():
    if "model" not in _model_cache:
        import os
        os.environ["HF_HOME"] = "/tmp/hf_cache"
        os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
        os.makedirs("/tmp/hf_cache", exist_ok=True)
        
        model = MoiraiModule.from_pretrained("Salesforce/moirai-1.0-R-small")
        model.eval()
        proj = torch.nn.Linear(model.config.d_model, D_OUT)
        proj.eval()
        _model_cache["model"] = model
        _model_cache["proj"] = proj
    return _model_cache["model"], _model_cache["proj"]

def get_embedding(flat_vector):
    model, proj = get_model()
    matrix = np.array(flat_vector, dtype=np.float32).reshape(SEQ_LEN, 6)
    tensor = torch.tensor(matrix).unsqueeze(0)  # (1, 30, 6)
    
    with torch.no_grad():
        encoder_out = model.encoder(tensor)  # (1, 30, d_model)
        pooled = encoder_out.mean(dim=1)     # (1, d_model)
        embedding = proj(pooled)             # (1, 256)
    
    return embedding.squeeze(0).numpy().tolist()

@pandas_udf(ArrayType(FloatType()))
def embed_trajectory(flat_series: pd.Series) -> pd.Series:
    return flat_series.apply(get_embedding)



In [0]:
df = df.withColumn("embedding", embed_trajectory("traj_flat"))

In [0]:
display(df.select("hex", "state", "window", "embedding").limit(5))

In [0]:
# from pyspark.sql.functions import avg, count

# agg_df = delay_df.groupBy("flight").agg(
#     avg("gs").alias("avg_speed"),
#     avg("alt_baro").alias("avg_altitude"),
#     avg("is_delayed").alias("delay_ratio"),
#     count("*").alias("record_count")
# )

In [0]:

import time
unique_id = int(time.time())
checkpoint_path = f"dbfs:/Volumes/pyspark/stream/sparkstreaming/checkpoints/congestion_{unique_id}"

query = agg_df.writeStream \
    .format("memory") \
    .queryName("debug_view") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .outputMode("update") \
    .start()

# 2. Wait for the stream to finish processing
query.awaitTermination()
display(spark.sql("SELECT * FROM debug_view"))

In [0]:
from pyspark.sql.window import Window as W
w_spec = W.partitionBy("hex").orderBy("event_time")
df = df.withColumn("prev_track", lag("track").over(w_spec))
       .withColumn("dt_sec", (unix_timestamp("event_time") - lag(unix_timestamp("event_time")).over(w_spec)))
       .withColumn("turn_rate",
           when(col("dt_sec") > 0,
               abs(col("track") - col("prev_track")) / col("dt_sec"))
           .otherwise(lit(0.0))
       )
# heading wraps at 360° — handle the 350° → 10° case:
       .withColumn("turn_rate",
           when(col("turn_rate") > 180, lit(360) - col("turn_rate"))  # shortest arc
           .otherwise(col("turn_rate"))
       )

In [0]:

import time
unique_id = int(time.time())
checkpoint_path = f"dbfs:/Volumes/pyspark/stream/sparkstreaming/checkpoints/congestion_{unique_id}"

query = agg_df.writeStream \
    .format("memory") \
    .queryName("debug_view") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .outputMode("append") \
    .start()

# 2. Wait for the stream to finish processing
query.awaitTermination()
display(spark.sql("SELECT * FROM debug_view"))